In [ ]:
import csv
import json
import logging
from openpyxl import Workbook

logging.basicConfig(
    filename="errors.log",
    level=logging.ERROR,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

def read_csv_generator(filename):

    with open(filename, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            yield row

class DataPipeline:

    def __init__(self, filename):

        self.filename = filename
        self.clean_data = []
        self.total_rows = 0
        self.valid_rows = 0
        self.invalid_rows = 0
        self.total_sales = 0
        self.generator = None

    def load(self):

        self.generator = read_csv_generator(self.filename)
        return self

    def clean(self):

        for row in self.generator:
            self.total_rows += 1
            try:
                product = row["Product"].strip()
                quantity = int(row["Quantity"])
                price = float(row["Price"])
                city = row["City"].strip()
                sales = quantity * price
                cleaned_row = {
                    "Product": product,
                    "Quantity": quantity,
                    "Price": price,
                    "City": city,
                    "Sales": sales
                }

                self.clean_data.append(cleaned_row)
                self.valid_rows += 1
                self.total_sales += sales

            except Exception as e:
                self.invalid_rows += 1
                logging.error(
                    f"Row {self.total_rows}: {row} | Error: {e}"
                )
        return self

    def export_csv(self, output_file):

        with open(output_file, "w", newline="", encoding="utf-8") as file:

            fieldnames = [
                "Product",
                "Quantity",
                "Price",
                "City",
                "Sales"
            ]

            writer = csv.DictWriter(
                file,
                fieldnames=fieldnames
            )

            writer.writeheader()
            writer.writerows(self.clean_data)

        return self

    def export_json(self, output_file):

        summary = {
            "Total Rows": self.total_rows,
            "Valid Rows": self.valid_rows,
            "Invalid Rows": self.invalid_rows,
            "Total Sales": self.total_sales,
            "Average Sales":
                round(
                    self.total_sales / self.valid_rows,
                    2
                )
                if self.valid_rows != 0 else 0
        }

        with open(output_file, "w") as file:
            json.dump(
                summary,
                file,
                indent=4
            )
        return self